In [ ]:
import pandas as pd
import numpy as np
from pyod.models.ecod import ECOD
import json


In [ ]:
# 1. Load your saved CSV
df = pd.read_csv("ai4i2020.csv")

sensor_cols = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear"
]

In [ ]:
# 2. Split into Normal (to fit baseline) and Faults (to test fault injection)
normal_data = df[df["Machine failure"] == 0][sensor_cols]
hdf_faults = df[df["HDF"] == 1][sensor_cols] # Heat Dissipation Failures

In [ ]:
# 3. Fit an unsupervised ECOD Anomaly Model on normal telemetry
clf = ECOD(contamination=0.03) # Expect ~3% anomalies threshold
clf.fit(normal_data)

# Calculate healthy baseline statistics for our Explainer
baseline_mean = normal_data.mean().values
baseline_std = normal_data.std().values

In [ ]:

# 4. Simulate receiving 1 faulty reading (Injected Heat Dissipation Fault)
sample_row = hdf_faults.iloc[0].values

# Get model prediction (0 = normal, 1 = anomaly) and probability score
is_anomaly = int(clf.predict([sample_row])[0])
anomaly_score = float(clf.predict_proba([sample_row])[0][1])

# Calculate Z-score per sensor to explain WHICH sensor went crazy
z_scores = np.abs((sample_row - baseline_mean) / baseline_std)
ranked_indices = np.argsort(z_scores)[::-1] # Sort highest deviation first

primary_sensor = sensor_cols[ranked_indices[0]]
secondary_sensor = sensor_cols[ranked_indices[1]]

In [ ]:
# 5. Build Deliverable 3 Output JSON for the Agent (Service 4)
alert_payload = {
    "machine_id": "CNC_MILL_01",
    "is_anomaly": bool(is_anomaly),
    "anomaly_score": round(anomaly_score, 3),
    "explanation": {
        "primary_trigger": primary_sensor,
        " primary_z_score": round(float(z_scores[ranked_indices[0]]), 2),
        "secondary_trigger": secondary_sensor,
        "secondary_z_score": round(float(z_scores[ranked_indices[1]]), 2),
        "summary": f"Abnormal pattern detected: {primary_sensor} and {secondary_sensor} deviated significantly from normal operating profile."
    },
    "supporting_context": {
        "current_telemetry": dict(zip(sensor_cols, [round(float(x), 2) for x in sample_row])),
        "normal_baseline_mean": dict(zip(sensor_cols, [round(float(x), 2) for x in baseline_mean]))
    }
}

print(json.dumps(alert_payload, indent=2))